<a href="https://colab.research.google.com/github/ChenOnline/APEX_WVC/blob/main/Confidence_Interval_1proportion_WVC_asc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STATS Confidence Intervals for 1 Population Proportion
Background adopted from David Schuster and Michelle Baca Reinke
The data set was curated by Ken Brown.

Licensed under CC BY-NC-SA

## Part I. Background

Inferential statistics allow us to make generalizations about a population by studying a sample (a subset of the population). Population parameters, such as the population proportion ($p$), are "true" characteristics of the entire population. These values are typically unknown. This is where we use inferential statistics to estimate these parameters from the sample data, enabling us to draw conclusions about a population.

However, a sample may not perfectly represent the entire population. **Confidence intervals allow us to quantify the precision and certainty of our estimates by providing a range of plausible values for the "true" population parameter**.

The following background is recommended before starting this module:

1. **Sampling, Sample Distributions, and Central Limit Theorem:** This module will work best if you have already covered the concepts of sampling, sample distributions, and central limit theorem in your course. You may want to review your notes or textbook before starting this module.

2. **Welcome to Colab:** If you have not worked through the <a href="https://colab.research.google.com/drive/1zTk_n6BL8Tvdhaufq_FYNXK0dbvGKnxX?usp=sharing">Welcome to Colab</a> notebook yet, you may want do that first.

## Part II. Activity

This activity will use a set of data of flights from the San Francisco Bay Area (SFO, OAK and SJC) to Seattle, Washington in 2014.


Set up this activity by running the code block below. This will import the data. Reminder: to run the cell, you can either use `Shift` + `Enter`, or you can hit the play button.

### 1. Get Data

Before you can begin these exercises, you need to run the code cell below, which will import the World Bank file and create a dataframe (i.e., spreadsheet) named `data`. After you run the cell, you will proceed to the next section to  preview the dataframe. Take note that it contains several columns (these will be described in the exercise).

###<font color="red">➡</font> Run the Cell to Set Up Example Data ###

To run the cell, click on it, and then you can either simultaneously hit `Shift` + `Enter`, or you can click the play button to the left of the cell.

After you click it, you should see the text "The data were loaded" underneath the code cell. If you see that, continue to the next section. If you come back to this notebook later, you will need to rerun this cell to load the data again.

If you see the text "There was a problem loading the data," then the most likely explanation is a bug that is our fault. Let your instructor know the notebook is not working properly.

In [ ]:
# Import library
import pandas as pd

# Read data file: World Bank
data = pd.read_csv('https://raw.githubusercontent.com/ChenOnline/data/refs/heads/main/Sample1_from_OnTimeBayAreaSEA2014.csv')

# Handle errors
try:
    data
    print('The data were loaded.')
except NameError:
    print('There was a problem loading the data.')

###<font color="red">➡</font> Run the Cell to Preview Data ###

Next, we will generate a quick preview of the data you just loaded. Run the cell that follows. To run the cell, click on it, and then you can either simultaneously hit `Shift` + `Enter`, or you can click the play button to the left of the cell.

After you run it, you should see a table with rows and columns. Each row is a country, with names listed in column `y` and mean life expectency (years) at birth in 2018 in column`x`.

In [ ]:
# Preview Data the first few rows of data
data.head()
# What do you think the following line of code would do?  Uncomment the code and run it to see for yourself.
#data.tail() ##there are 299 rows!

Run the following code to draw your own random sample of n=100


In [ ]:
# Draw a random sample of 100
random_seed = int(input("Please enter the last three digits of your G number: "))

mysample = data.sample(n=100, replace=False, random_state=random_seed)

mysample


### <font color="red">➡</font> Answer the Following Questions ###
Text like this is also in a cell. Double click right here to edit this text cell. Then, type your answers to each question below the question. When finished, press `Shift` + `Enter` and you will see your answers in the notebook.

-Q1. What is the population this sample is drawn from?
*  

-Q2. What kind of study is this?  Experimental or observational? And why is this important?
*  

### 2. Explore the Data

Now that you have seen a preview, it is time to explore the dataframe! We will be using the following format to refer to each variable:     
`name of dataframe['column name']`

The following are a few of the variables we will explore:

- `mysample['ORIGIN']`: airport code for where the flight oringinated: SJC (San Jose), SFO (San Francisco), and OAK (Oakland)
- `mysample['DepDelay15']`: 0 if the flight was not at least 15 minutes late taking off and 1 if the flight was at least 15 minutes late taking off
- `mysample['DepDelayOver15']`: Whether the flight was more than 15 minutes late taking off: "Departure Delay 15 min or less", "Departure Delay more than 15 min"


### <font color="red">➡</font> Run the Cell to Examine a Variable ###      



In [ ]:
# Let's look at the ORIGINating airports
mysample['ORIGIN'].describe()

In [ ]:
# Now run the block of code below to see a summary of ORIGINating airports.
# build a frequency table
origin_counts = mysample['ORIGIN'].value_counts()


# Calculate percentages, also called relative frequency
origin_percentages = mysample['ORIGIN'].value_counts(normalize=True) * 100

# Create a new DataFrame, formatting the 'Percentage' (relative frequency) column
frequency_table = pd.DataFrame({
    'Count': mysample['ORIGIN'].value_counts(), # build a frequency table
    'Percentage': (mysample['ORIGIN'].value_counts(normalize=True) * 100).map('{:.2f}%'.format) # Format percentages
})

# Add a total row
frequency_table.loc['Total'] = pd.Series({'Count': frequency_table['Count'].sum(), 'Percentage': '100.00%'})

# Display the frequency (and relative frequency) table
print(frequency_table)

### 3. Confidence Intervals (a.k.a. Interval Estimation)

Up to this point, we have been using single numbers to summarize our sample data. We call this **point estimation**, as in a single data point. Point estimation refers to estimating a population parameter, such as the population proportion. For example, using the sample proportion $\hat{p}$ (calculated from sample data) to estimate the population proportion $p$ (the group of interest).    

A larger sample size gives a more reliable estimate, and the central limit theorem explains why. The central limit theorem says that an increase to the sample size will reduce the standard error. It says so in the standard error formula. But if we only report a point estimate, we have no way to communicate how good the point estimate is. We could include the sample size, and that would be helpful. There is another way of expressing this information; we can report a **confidence interval**.

A confidence interval is a range of values around a sample datapoint (e.g., $\hat{p}$) where we can be relatively certain that the true population value (e.g., $p$) lies. An interval refers to a range of values, defined by a lower bound and an upper bound. Therefore, reporting a confidence interval means giving two values. However, as with the central limit theorem, the logic underneath the confidence interval can be a bit tricky.    

### 4. Logic of Confidence Intervals

Theoretically, if a large number of samples are drawn, with a confidence interval computed for each sample, a 95% confidence level implies that we can expect that approximately 95% of these confidence intervals will contain the population parameter. That is, if you computed 100 confidence intervals from 100 different samples, you would expect that the true population parameter would be captured by 95 of those confidence intervals.

Thus, a confidence interval provides information about what would happen if we took many samples. A good way to say this that will get researchers to nod is that *confidence intervals tell you about replication.*

### 5. Common Misinterpretation of Confidence Levels

When we do research, we do not usually take repeated samples. We usually only take one sample. Yet, the logic of confidence intervals is a probability statement that imagines we took repeated random samples. In all, this can make the logic of confidence intervals a bit confusing. You will hear people misinterpret this concept.

A common misinterpretation is a statement such as this: "There is a 95% *chance* that the true population mean lies within a single 95% confidence interval".

This is not correct because true population mean either *is* or *is not* within that range, and we have no way of knowing the truth. Instead, we imagine the liklihood of repeated samples and their corresponding confidence intervals containing the true population parameter.

Finally, you can compute other levels of confidence, but the 95% level of confidence is the most commonly reported value.



### 6. Sample Statistics

Before we learn to calculate a confidence interval, we need to determine the sample size, sample statistic such as sample proportion, and the mean and standard deviation (SE) of the sampling distribution. These values are used to calculate our confidence interval.

### 7. Calculating the Confidence Interval for One Population Proportion

To calculate a confidence interval, we need to first calculate the standard error, and then calculate the lower and upper bounds (also called the lower limit and upper limit) of our confidence interval. For a 95% confidence level, the lower bound is derived by subtracting 1.96 times the standard error, which is the standard deviation of the sampling distribution, from the sample statistic, while the upper bound is obtained by adding the same value to the sample statistic.

#### Standard Error

In a previous module, you learned how to caculate the standard error. As a reminder, the standard error is calculated using this formula:

$\displaystyle SE=\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$

As sample size, $n$ increases, SE decreases, since $n$ is in the denominator.

#### Lower Bound

Formula for the lower bound of the 95% confidence interval:

$\displaystyle \hat{p} -1.96 SE$  

In words, the lower bound of the 95% confidence interval is equal to the sample proportion minus 1.96 standard errors.

#### Upper Bound

Formula for the upper bound of the 95% confidence interval:

$\displaystyle \hat{p} + 1.96 SE$  

In words, the upper bound of the 95% confidence interval is equal to sample proportion plus 1.96 standard errors.


###<font color="red">➡</font> Run the Cell to Calculate the Confidence Interval ###

We use the sample in the data set we read in the first coding cell to practice finding a confidence interval for the true population proportion.

Let's look at **proportion of those flights that were delayed for over 15 minutes on departure.** Find a 95% confidence interval for your random sample.

To run the cell, click on it, and then you can either simultaneously hit `Shift` + `Enter`, or you can click the play button to the left of the cell.

In [ ]:
# Import libraries
import numpy as np

#confidence level
zstar = 1.96  # for a confidence level of 95%

# Sample size
nn = len(mysample)

# sample statistic
# Let's find the sample proportion of those flights that were delayed for over 15 minutes on departure
# number of "successes" (flights delayed at least 15 minutes on departure)
xx = (mysample['DepDelay15'] == 1).sum()

# compute the sample statistic, p_hat, an unbiased estimator of the population paramenter, p
p_hat = xx/nn

# Calculate SE, the SD for the sampling distribution
SE = np.sqrt(p_hat*(1-p_hat)/nn)

# Compute the CI
ll = p_hat - zstar*SE
ul = p_hat + zstar*SE

# Calculate confidence interval
print('number of success =', xx, '\nsample size =', nn, '\np_hat=', round(p_hat, 4), '\nSE =', round(SE, 5),
      '\nThe confidence interval is (', round(ll, 3), ',', round(ul, 3), ')', sep=' ')

**Visualize the Confidence Interval**

In [ ]:
# Import libraries
from scipy import stats
import matplotlib.pyplot as plt

# number of digigs to round for easy reading
ndigits = 3

# Specify confidence level
conf_level = 0.95

# Uncomment the following lines of code and fill in your numbers, if you need to calculate p_hat and SE
# nn =                                             #sample size
# p_hat =                                          #sample proportion
# SE = np.sqrt(p_hat*(1-p_hat)/nn)                 #SE

# Calculate confidence interval
ci = stats.norm.interval(conf_level, loc=p_hat, scale=SE)

# Print Results
print(
    f"""Confidence Interval:  ({ci[0]:.3f}, {ci[1]:.3f})
    Lower bound = {round(ci[0],ndigits)}
    Upper bound = {round(ci[1],ndigits)}
    """
    )

# Visualize the single confidence interval and the population proportion
plt.figure(figsize=(10, 2)) # Adjust figure size for a single CI

# Plot the single confidence interval as a horizontal line
plt.plot([ci[0], ci[1]], [0, 0], color='gray', linewidth=3, alpha=0.8)
# Plot the sample proportion as a dot (at y=0)
plt.plot(p_hat, 0, 'o', color='blue', markersize=8, label='Sample Proportion (p_hat)')

plt.xlabel("Proportion of Flights Delayed Over 15 Minutes")
plt.ylabel("") # No y-label needed for a single interval
plt.title("Single Confidence Interval Visualization")
plt.yticks([]) # Hide y-axis ticks for cleaner visualization
plt.ylim(-0.5, 0.5) # Set y-axis limits to center the single interval
plt.xlim(0.05, 0.4) # Set x-axis limits as requested
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

**Reporting Results**   
### <font color="red">➡</font> Answer the Following Questions ###
Text like this is also in a cell. Double click right here to edit this text cell. Then, type your answers to each question below the question. When finished, press `Shift` + `Enter` and you will see your answers in the notebook.  

-Q3.  Write your confidence interval in the format of (lower bound, upper bound) below.
*  


-Q4. Our variable is `DepDelay15`: 0 if the flight was not at least 15 minutes late taking off and 1 if the flight was at least 15 minutes late taking off.  And this sample data set has flight information from 2014 between SF bay area and Seattle, Washington.  Write a sentence, or two, to interpret the confidence interval you found in context.  Be sure to include the confidence level, population of interest, the lower and upper bounds of the confidence interval, and context.

*  

In this example, we actually have the population.  So we can find the actual population propportion.  Run the following code cell to find the population proportion.

In [ ]:
population_proportion = (data['DepDelay15'] == 1).sum() / len(data)
print(f"Population Proportion: {population_proportion:.4f}")

### <font color="red">➡</font> Answer the Following Questions ###
-Q5 What is the population proportion of flights between the bay area and Seattle that was delayed at least 15 minutes on take off?

*

-Q6 Now that you know the true population proportion, did your confidence interval built using a random sample and p_hat, contain the population proportion, p?  If p was included in the confidence interval, was it in the middle?

*


Now let's understand the confidence level.  Run the following code to generate 100 random samples.  With each sample, we will use its p_hat to build a confidence interval (CI).  These CIs are graphed below along with the true population proportion.  Please pay attention to the few orange colored confidence intervals.  Count how many there are.  Run the code cell again with a confidence level of 0.90 and see how that affects the number of these orange colored CIs.  What about a confidence level of 0.99?

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# Choose Confidence level, you can adjust
conf_level = 0.90

# Sanple size, you can adjust
nn = 100

# number of samples, you can adjust
n_samples = 100

# Step 1: Define the population proportion (assuming this is already done in a previous cell, but including here for completeness)
# For this simulation, we'll assume the proportion from the entire dataset is the population proportion
population_proportion = (data['DepDelay15'] == 1).sum() / len(data)
print(f"Population Proportion: {population_proportion:.4f}")

# Step 2: Create a function to draw a random sample and calculate the confidence interval
def calculate_confidence_interval(data, sample_size, conf_level=conf_level):
    """Draws a random sample, calculates the sample proportion and its confidence interval."""
    # Draw random sample
    sample = data.sample(n=sample_size, replace=True)

    # Calculate sample proportion
    sample_proportion = (sample['DepDelay15'] == 1).mean()

    # Calculate standard error
    SE = np.sqrt(sample_proportion * (1 - sample_proportion) / sample_size)

    # Calculate confidence interval using scipy.stats.norm.interval
    ci = stats.norm.interval(conf_level, loc=sample_proportion, scale=SE)

    return sample_proportion, ci

# Step 3: Draw 100 samples and calculate their confidence intervals
sample_size = nn

confidence_intervals = []
sample_proportions = []

for _ in range(n_samples):
    sample_proportion, ci = calculate_confidence_interval(mysample, sample_size)
    sample_proportions.append(sample_proportion)
    confidence_intervals.append(ci)

# Step 4: Visualize the confidence intervals and the population proportion
plt.figure(figsize=(10, 8))

for i, ci in enumerate(confidence_intervals):
    # Determine the color based on whether the confidence interval contains the population proportion
    color = 'blue' if ci[0] <= population_proportion <= ci[1] else 'orange'

    # Plot the confidence interval as a horizontal line
    plt.plot([ci[0], ci[1]], [i, i], color=color, alpha=0.5)
    # Plot the sample proportion as a dot
    plt.plot(sample_proportions[i], i, 'o', color='red', markersize=4)

# Plot the true population proportion as a vertical line
plt.axvline(x=population_proportion, color='green', linestyle='--', label=f'Population Proportion ({population_proportion:.4f})')

plt.xlabel("Proportion of Flights Delayed Over 15 Minutes")
plt.ylabel("Sample Number")
plt.title(f"Confidence Intervals using {n_samples} Samples (with Sample Size = {sample_size})")
plt.yticks([]) # Hide y-axis ticks for cleaner visualization
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### <font color="red">➡</font> Answer the Following Questions ###
Note that a confidence interval (CI) either contains the true population proportion or not.  THE CONFIDENCE LEVEL IS NOT ABOUT A PARTICULAR CONFIDENCE INTERVAL BUT ABOUT THE PROCESS.  That is, when we repeat the process of drawing a random sample and constructing a CI using the sample, the confidence level tells us how many of those CIs contains p, or the population proportion.

-Q7 If we drew 200 random samples and constructed 200 CIs with these samples using a confidence level of 0.95, how many CIs do you expect to capture p?

*

-Q8 Why do you think some CIs do not capture p?

*


----
## Part IV. Summary

Confidence intervals are important in conveying the reliability and precision of population inferences based on samples.

* In this module, we discussed the logic underlying confidence intervals.
* We practiced calculation of sample statistics, standard error, and confidence intervals.
* Lastly, we practiced interpretation of the results.

---
## Part VI. All Done, Congrats!

Today you've not only learned about confidence intervals, but you've also learned about some Python code. High five!

<img src="https://live.staticflickr.com/3471/3904325807_8ab0190152_b.jpg" alt="High-five!" width="300"/>

["High-five!"](https://live.staticflickr.com/3471/3904325807_8ab0190152_b.jpg) by Nick J Webb is licensed under CC BY 2.0